In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import wbgapi as wb

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 7
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## IMF SPI Pipeline

**Source:** World Bank Statistical Performance Indicators
**Access:** Automated via `wbgapi` — no manual step required
**Download instructions:** See `docs/instructions_data_maintenance.md` — IMF_SPI section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| SPI Overall Score | Statistical and informational infrastructure | Primary tier 1 |
| SPI Pillar 1: Data Use | Statistical and informational infrastructure | Primary tier 1 |
| SPI Pillar 2: Data Services | Statistical and informational infrastructure | Primary tier 1 |
| SPI Pillar 3: Data Products | Statistical and informational infrastructure | Primary tier 1 |
| SPI Pillar 4: Data Sources | Statistical and informational infrastructure | Primary tier 1 |
| SPI Pillar 5: Data Infrastructure | Statistical and informational infrastructure | Primary tier 1 |

In [2]:
import wbgapi as wb
import pandas as pd
from datetime import datetime

# SPI indicators — overall score plus 5 pillars
SPI_INDICATORS = {
    'IQ.SPI.OVRL': 'spi_overall',
    'IQ.SPI.PIL1': 'spi_pillar1_data_use',
    'IQ.SPI.PIL2': 'spi_pillar2_data_services',
    'IQ.SPI.PIL3': 'spi_pillar3_data_products',
    'IQ.SPI.PIL4': 'spi_pillar4_data_sources',
    'IQ.SPI.PIL5': 'spi_pillar5_data_infrastructure',
}

# Apply SSL setting
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    wb.fetch_options = {'verify': False}

# Set WDI database — SPI is in WDI (db=2)
wb.db = 2

print("Fetching SPI data...")
spi_raw = wb.data.DataFrame(
    list(SPI_INDICATORS.keys()),
    time=range(FRAMEWORK_START_YEAR, CURRENT_YEAR + 1),
    labels=True
)

print(f"Raw shape: {spi_raw.shape}")
print(spi_raw.head())

Fetching SPI data...
Raw shape: (1596, 38)
                                   Country  \
economy series                               
ZWE     IQ.SPI.OVRL               Zimbabwe   
ZMB     IQ.SPI.OVRL                 Zambia   
YEM     IQ.SPI.OVRL            Yemen, Rep.   
PSE     IQ.SPI.OVRL     West Bank and Gaza   
VIR     IQ.SPI.OVRL  Virgin Islands (U.S.)   

                                                                Series  \
economy series                                                           
ZWE     IQ.SPI.OVRL  Statistical performance indicators (SPI): Over...   
ZMB     IQ.SPI.OVRL  Statistical performance indicators (SPI): Over...   
YEM     IQ.SPI.OVRL  Statistical performance indicators (SPI): Over...   
PSE     IQ.SPI.OVRL  Statistical performance indicators (SPI): Over...   
VIR     IQ.SPI.OVRL  Statistical performance indicators (SPI): Over...   

                     YR1990  YR1991  YR1992  YR1993  YR1994  YR1995  YR1996  \
economy series                      

In [3]:
# Reset index and melt to long format
spi = spi_raw.reset_index()
spi = spi.rename(columns={'economy': 'country_code', 'series': 'indicator_code'})
spi['indicator_name'] = spi['indicator_code'].map(SPI_INDICATORS)

year_cols = [c for c in spi.columns if c.startswith('YR')]
spi_long = spi.melt(
    id_vars=['country_code', 'Country', 'indicator_code', 'indicator_name'],
    value_vars=year_cols,
    var_name='year_str',
    value_name='value'
)
spi_long['year'] = spi_long['year_str'].str.replace('YR', '').astype(int)
spi_long = spi_long.drop(columns=['year_str', 'indicator_code'])
spi_long = spi_long.rename(columns={'Country': 'country_name'})

# Remove rows with no data
spi_long = spi_long.dropna(subset=['value'])

# Pivot to wide format
spi_wide = spi_long.pivot_table(
    index=['country_code', 'country_name', 'year'],
    columns='indicator_name',
    values='value'
).reset_index()

spi_wide.columns.name = None
spi_wide = spi_wide.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {spi_wide.shape}")
print(f"Years: {spi_wide['year'].min()} — {spi_wide['year'].max()}")
print(f"Countries: {spi_wide['country_code'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (spi_wide.isnull().sum() / len(spi_wide) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(spi_wide.head())

Shape: (4593, 9)
Years: 2004 — 2024
Countries: 221

Missing values (%):
spi_overall                        64.1
spi_pillar2_data_services          63.8
spi_pillar5_data_infrastructure    62.0
spi_pillar4_data_sources           61.1
spi_pillar3_data_products           5.6
dtype: float64
  country_code country_name  year  spi_overall  spi_pillar1_data_use  \
0          ABW        Aruba  2004          NaN                   0.0   
1          ABW        Aruba  2005          NaN                   0.0   
2          ABW        Aruba  2006          NaN                   0.0   
3          ABW        Aruba  2007          NaN                   0.0   
4          ABW        Aruba  2008          NaN                   0.0   

   spi_pillar2_data_services  spi_pillar3_data_products  \
0                        NaN                        NaN   
1                        NaN                   19.83125   
2                        NaN                   19.03125   
3                        NaN                

In [4]:
# Get metadata from API automatically
wb.db = 2
wdi_source_meta = next(s for s in wb.source.list() if s['id'] == '2')
data_as_of_date = wdi_source_meta['lastupdated'][:7]
latest_year = str(int(spi_wide['year'].max()))

print(f"Data as of: {data_as_of_date}")
print(f"Latest year: {latest_year}")

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "spi_clean.csv")
spi_wide.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {spi_wide.shape}")

# Update download log
update_entry(
    "IMF_SPI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="spi_clean.csv",
    latest_available_version=latest_year,
    notes="6 SPI indicators — overall score plus 5 pillars. Full coverage from 2016; partial from 2004. 221 countries."
)

print_entry("IMF_SPI")

Data as of: 2026-04
Latest year: 2024
Written: /Users/boulanger/Documents/governance-framework/data/processed/spi_clean.csv
Shape: (4593, 9)
[download_log] Updated entry for IMF_SPI
  source_id: IMF_SPI
  last_attempted_date: 2026-05-31
  last_successful_download_date: 2026-05-31
  data_as_of_date: 2026-04
  local_filename: spi_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: 6 SPI indicators — overall score plus 5 pillars. Full coverage from 2016; partial from 2004. 221 countries.
